# TISER Baseline — Colab SMOKE TEST

Validates the full pipeline end-to-end (fetch → train → adapter → eval → metrics) on a **free T4**, using `config/config_smoke.yaml`: a tiny 1.5B model, 300 training examples, 1 epoch, sampled eval.

**This run is about plumbing, not accuracy.** Expect bad EM/F1 — that is fine. Success = the pipeline completes, `metrics.json` is produced, malformed count is low, and the sanity-check cell confirms loss masking + EOS.

In [ ]:
!nvidia-smi

In [ ]:
# Fresh Colab: clone the repo and cd into it.
# (Remove this cell if you opened the notebook from an existing checkout.)
!git clone https://github.com/thestbobo/tiser_temporal_reasoning_extension.git
%cd tiser_temporal_reasoning_extension

In [ ]:
# Use Colab's bundled torch (do NOT reinstall it — it breaks bitsandbytes).
!pip install -r requirements.txt
!pip install -e .

In [ ]:
# On Colab the local POLITO path is absent, so this falls back to the HF mirror / git-lfs.
!python scripts/fetch_data.py --config config/config_smoke.yaml

## Sanity check — the two RED problems (run BEFORE training)

This builds the trainer exactly as `train.py` does and inspects one collated batch.

- **RED A (completion-only loss):** the text the loss is computed on should be **only the `<reasoning>…<answer>` trace**, NOT the question/context prompt. If the prompt text shows up below, masking is OFF → the fix is to switch to a single `text` field + `DataCollatorForCompletionOnlyLM`.
- **RED B (EOS):** the sequence should end with the EOS token id so the model learns to stop.
- If this cell **errors** with a dataset-format complaint, that itself is finding A (TRL 0.12.2 not accepting the prompt/completion columns).

In [ ]:
from src.utils.config import load_config
from src.model.loader import load_model_and_tokenizer
from src.data.dataset import build_train_dataset
from src.train.trainer import build_trainer

cfg = load_config('config/config_smoke.yaml')
model, tok = load_model_and_tokenizer(cfg)
# build_train_dataset now returns a *pretokenized* dataset (input_ids/labels) with
# chat-template wrapping + completion-only masking already applied.
ds, _ = build_train_dataset(cfg.paths.train_file, tok, cfg.train.max_seq_len, 8)
trainer = build_trainer(cfg, model, tok, ds)

batch = next(iter(trainer.get_train_dataloader()))
labels = batch['labels'][0]
input_ids = batch['input_ids'][0]
n_total = labels.numel()
n_masked = int((labels == -100).sum())
print(f'tokens: {n_total} | masked(-100): {n_masked} | trained-on: {n_total - n_masked}')

# RED A: what text is the loss actually computed on? (should be ONLY the trace + EOS)
unmasked_ids = input_ids[labels != -100]
print('\n--- text the loss is computed on (should be the <reasoning>..<answer> trace) ---')
print(tok.decode(unmasked_ids)[:800])

# RED B: does the sequence end with EOS so the model learns to stop?
print('\nEOS id:', tok.eos_token_id, '| last 5 input_ids:', input_ids[-5:].tolist())
print('EOS present in sequence:', tok.eos_token_id in input_ids.tolist())

## Train (smoke)
All the small-run knobs (1.5B model, `subset_size: 300`, 1 epoch) live in `config/config_smoke.yaml`, so no CLI flags are needed.

In [ ]:
!python scripts/train.py --config config/config_smoke.yaml

In [ ]:
!python scripts/evaluate.py --config config/config_smoke.yaml

In [ ]:
import json, yaml

run_name = yaml.safe_load(open('config/config_smoke.yaml'))['run_name']
metrics = json.load(open(f'outputs/{run_name}/metrics.json'))

print(f"macro-EM {metrics['macro_em']:.3f} | macro-F1 {metrics['macro_f1']:.3f}")
print(f"malformed: {metrics['n_malformed']}/{metrics['n_total']}\n")
for split, m in metrics['per_split'].items():
    print(f"{split:16s} EM {m['em']:.3f}  F1 {m['f1']:.3f}  (n={m['n']})")

In [ ]:
# Inspect a few predictions: raw generation, parsed answer, gold, EM/F1.
with open(f'outputs/{run_name}/predictions.jsonl') as f:
    for line in list(f)[:3]:
        r = json.loads(line)
        print(f"[{r['dataset_name']}] gold={r['gold']!r} pred={r['pred_answer']!r} "
              f"em={r['em']} f1={r['f1']:.2f} malformed={r['malformed']}")
        print('  raw:', r['raw_generation'][:300].replace('\n', ' '), '\n')

## Smoke-test pass criteria

- [ ] Sanity-check cell ran: loss is computed **only on the trace** (prompt masked), and EOS is present.
- [ ] Training completed and saved an adapter under `model/tiser_smoke/adapter`.
- [ ] `outputs/tiser_smoke/metrics.json` exists with per-split EM/F1 (values will be poor — expected).
- [ ] `n_malformed` is low (parser and generation format agree).

If all four hold, the pipeline is wired correctly — move to the full run on RunPod via `notebooks/colab_run.ipynb` / `config/config.yaml` (or an A100/H100 RunPod box).